<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Multi-Target/Multi_Target_3_Iterations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
path = "/content/drive/MyDrive/Research v2/Research 18.03.2026/GlobalWeatherRepository (3).csv"
df = pd.read_csv(path)

print("Original shape:", df.shape)
df.head()

Original shape: (130003, 41)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


In [3]:
targets = [
    "air_quality_PM2.5",
    "temperature_celsius",
    "humidity"
]

print("Targets:", targets)

Targets: ['air_quality_PM2.5', 'temperature_celsius', 'humidity']


In [4]:
drop_cols_manual = [
    "temperature_fahrenheit",
    "feels_like_fahrenheit",
    "wind_kph",
    "gust_kph",
    "pressure_in",
    "precip_in",
    "visibility_miles",
    "air_quality_us-epa-index",
    "air_quality_gb-defra-index"
]

existing_drop_cols = [col for col in drop_cols_manual if col in df.columns]

df = df.drop(columns=existing_drop_cols)

print("Dropped manually:", existing_drop_cols)
print("Shape after manual drop:", df.shape)

Dropped manually: ['temperature_fahrenheit', 'feels_like_fahrenheit', 'wind_kph', 'gust_kph', 'pressure_in', 'precip_in', 'visibility_miles', 'air_quality_us-epa-index', 'air_quality_gb-defra-index']
Shape after manual drop: (130003, 32)


In [5]:
# Convert timestamp if present
if "last_updated" in df.columns:
    df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")
    df = df.dropna(subset=["last_updated"])
    df = df.sort_values("last_updated").reset_index(drop=True)

# Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)

# Fill missing numeric values with median
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill missing categorical values
cat_cols = df.select_dtypes(include=["object"]).columns
df[cat_cols] = df[cat_cols].fillna("Unknown")

print("Shape after cleaning:", df.shape)

Shape after cleaning: (130003, 32)


In [6]:
from sklearn.preprocessing import LabelEncoder

if "condition_text" in df.columns:
    le_condition = LabelEncoder()
    df["condition_text"] = le_condition.fit_transform(df["condition_text"])

if "wind_direction" in df.columns:
    le_wind = LabelEncoder()
    df["wind_direction"] = le_wind.fit_transform(df["wind_direction"])

print("Categorical encoding completed.")

Categorical encoding completed.


In [7]:
exclude_cols = targets.copy()

if "last_updated" in df.columns:
    exclude_cols.append("last_updated")

candidate_features = [col for col in df.columns if col not in exclude_cols]

print("Number of candidate features:", len(candidate_features))
print(candidate_features)

Number of candidate features: 28
['country', 'location_name', 'latitude', 'longitude', 'timezone', 'last_updated_epoch', 'condition_text', 'wind_mph', 'wind_degree', 'wind_direction', 'pressure_mb', 'precip_mm', 'cloud', 'feels_like_celsius', 'visibility_km', 'uv_index', 'gust_mph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM10', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination']


In [8]:
corr_matrix = df.corr(numeric_only=True)

for target in targets:
    corr_target = corr_matrix[target].drop(target).sort_values(key=np.abs, ascending=False)
    print(f"\nTop correlations for {target}:")
    print(corr_target.head(15))


Top correlations for air_quality_PM2.5:
air_quality_PM10                0.645805
air_quality_Carbon_Monoxide     0.615754
air_quality_Nitrogen_dioxide    0.523122
air_quality_Sulphur_dioxide     0.325861
humidity                       -0.210262
cloud                          -0.178813
visibility_km                  -0.117687
longitude                       0.103413
gust_mph                       -0.062819
condition_text                  0.059368
precip_mm                      -0.052686
temperature_celsius             0.050313
uv_index                        0.048521
wind_mph                       -0.041501
air_quality_Ozone               0.039541
Name: air_quality_PM2.5, dtype: float64

Top correlations for temperature_celsius:
feels_like_celsius              0.983635
uv_index                        0.488923
latitude                       -0.374355
humidity                       -0.343765
pressure_mb                    -0.285187
air_quality_Ozone               0.265507
condition_text 

In [9]:
selected_feature_set = set()

for target in targets:
    corr_target = corr_matrix[target].drop(target)
    useful = corr_target[abs(corr_target) >= 0.10].index.tolist()
    selected_feature_set.update(useful)

selected_feature_set = list(selected_feature_set)

# remove targets if they accidentally came in
selected_feature_set = [col for col in selected_feature_set if col not in targets]

print("Selected candidate features from target correlations:")
print(selected_feature_set)
print("Count:", len(selected_feature_set))

Selected candidate features from target correlations:
['visibility_km', 'last_updated_epoch', 'air_quality_Ozone', 'latitude', 'air_quality_PM10', 'condition_text', 'air_quality_Sulphur_dioxide', 'uv_index', 'air_quality_Nitrogen_dioxide', 'feels_like_celsius', 'precip_mm', 'cloud', 'pressure_mb', 'longitude', 'air_quality_Carbon_Monoxide']
Count: 15


In [10]:
X_selected = df[selected_feature_set].copy()

corr_selected = X_selected.corr(numeric_only=True).abs()

upper = corr_selected.where(
    np.triu(np.ones(corr_selected.shape), k=1).astype(bool)
)

to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.85)]

print("Highly correlated input features to drop (|corr| > 0.85):")
print(to_drop_corr)

X_final = X_selected.drop(columns=to_drop_corr)
print("Final input shape:", X_final.shape)

Highly correlated input features to drop (|corr| > 0.85):
[]
Final input shape: (130003, 15)


In [11]:
y_final = df[targets].copy()

multi_target_df = pd.concat([X_final, y_final], axis=1)

print("Final cleaned multi-target dataset shape:", multi_target_df.shape)
multi_target_df.head()

Final cleaned multi-target dataset shape: (130003, 18)


,visibility_km,last_updated_epoch,air_quality_Ozone,latitude,air_quality_PM10,condition_text,air_quality_Sulphur_dioxide,uv_index,air_quality_Nitrogen_dioxide,feels_like_celsius,precip_mm,cloud,pressure_mb,longitude,air_quality_Carbon_Monoxide,air_quality_PM2.5,temperature_celsius,humidity
0,16.0,1715849100,62.2,46.60,7.1,2,0.2,1.0,2.5,16.1,0.00,0,1012.0,-120.49,198.6,6.3,16.1,58
1,10.0,1715849100,23.3,14.10,25.3,32,1.4,1.0,3.7,25.3,0.28,37,1017.0,-87.22,377.2,19.0,23.0,78
2,10.0,1715849100,5.9,13.71,28.1,23,7.5,1.0,7.7,30.2,0.30,50,1010.0,-89.20,460.6,20.4,26.0,94
3,5.0,1715849100,0.4,14.62,178.1,19,19.3,1.0,35.0,20.0,0.09,100,1019.0,-90.53,2243.0,132.0,20.0,88
4,10.0,1715849100,34.0,17.25,32.1,30,0.2,1.0,0.3,29.6,0.00,94,1007.0,-88.77,307.1,7.7,26.0,89


In [12]:
multi_target_df.to_csv("cleaned_multi_target_dataset.csv", index=False)

from google.colab import files
files.download("cleaned_multi_target_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
split_index = int(0.8 * len(multi_target_df))

train_df = multi_target_df.iloc[:split_index].copy()
test_df = multi_target_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (104002, 18)
Test shape : (26001, 18)


In [14]:
n_train = len(train_df)

iter1 = train_df.iloc[:int(0.33 * n_train)].copy()
iter2 = train_df.iloc[int(0.33 * n_train):int(0.66 * n_train)].copy()
iter3 = train_df.iloc[int(0.66 * n_train):].copy()

print("Iteration 1 shape:", iter1.shape)
print("Iteration 2 shape:", iter2.shape)
print("Iteration 3 shape:", iter3.shape)

Iteration 1 shape: (34320, 18)
Iteration 2 shape: (34321, 18)
Iteration 3 shape: (35361, 18)


In [15]:
final_features = [col for col in multi_target_df.columns if col not in targets]

print("Final features used for multi-target model:")
print(final_features)
print("Feature count:", len(final_features))

def get_xy(data):
    X = data[final_features]
    y = data[targets]
    return X, y

X1, y1 = get_xy(iter1)
X2, y2 = get_xy(iter2)
X3, y3 = get_xy(iter3)
X_test, y_test = get_xy(test_df)

print(X1.shape, y1.shape)
print(X2.shape, y2.shape)
print(X3.shape, y3.shape)
print(X_test.shape, y_test.shape)

Final features used for multi-target model:
['visibility_km', 'last_updated_epoch', 'air_quality_Ozone', 'latitude', 'air_quality_PM10', 'condition_text', 'air_quality_Sulphur_dioxide', 'uv_index', 'air_quality_Nitrogen_dioxide', 'feels_like_celsius', 'precip_mm', 'cloud', 'pressure_mb', 'longitude', 'air_quality_Carbon_Monoxide']
Feature count: 15
(34320, 15) (34320, 3)
(34321, 15) (34321, 3)
(35361, 15) (35361, 3)
(26001, 15) (26001, 3)


In [16]:
for col in targets:
    multi_target_df[f"{col}_lag1"] = multi_target_df[col].shift(1)
    multi_target_df[f"{col}_lag2"] = multi_target_df[col].shift(2)

multi_target_df = multi_target_df.dropna().reset_index(drop=True)

print("Shape after adding target lag features:", multi_target_df.shape)
multi_target_df.head()

Shape after adding target lag features: (130001, 24)


,visibility_km,last_updated_epoch,air_quality_Ozone,latitude,air_quality_PM10,condition_text,air_quality_Sulphur_dioxide,uv_index,air_quality_Nitrogen_dioxide,feels_like_celsius,...,air_quality_Carbon_Monoxide,air_quality_PM2.5,temperature_celsius,humidity,air_quality_PM2.5_lag1,air_quality_PM2.5_lag2,temperature_celsius_lag1,temperature_celsius_lag2,humidity_lag1,humidity_lag2
0,10.0,1715849100,5.9,13.71,28.1,23,7.5,1.0,7.7,30.2,...,460.6,20.4,26.0,94,19.0,6.3,23.0,16.1,78.0,58.0
1,5.0,1715849100,0.4,14.62,178.1,19,19.3,1.0,35.0,20.0,...,2243.0,132.0,20.0,88,20.4,19.0,26.0,23.0,94.0,78.0
2,10.0,1715849100,34.0,17.25,32.1,30,0.2,1.0,0.3,29.6,...,307.1,7.7,26.0,89,132.0,20.4,20.0,26.0,88.0,94.0
3,10.0,1715849100,14.3,12.15,14.7,41,11.4,1.0,6.5,30.6,...,500.7,11.7,27.2,80,7.7,132.0,26.0,20.0,89.0,88.0
4,7.0,1715849100,0.0,9.97,23.3,4,6.6,1.0,10.9,21.0,...,1161.6,21.7,21.0,100,11.7,7.7,27.2,26.0,80.0,89.0


In [17]:
split_index = int(0.8 * len(multi_target_df))

train_df = multi_target_df.iloc[:split_index].copy()
test_df = multi_target_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (104000, 24)
Test shape : (26001, 24)


In [18]:
n_train = len(train_df)

iter1 = train_df.iloc[:int(0.33 * n_train)].copy()
iter2 = train_df.iloc[int(0.33 * n_train):int(0.66 * n_train)].copy()
iter3 = train_df.iloc[int(0.66 * n_train):].copy()

print("Iteration 1 shape:", iter1.shape)
print("Iteration 2 shape:", iter2.shape)
print("Iteration 3 shape:", iter3.shape)

Iteration 1 shape: (34320, 24)
Iteration 2 shape: (34320, 24)
Iteration 3 shape: (35360, 24)


In [19]:
final_features = [col for col in multi_target_df.columns if col not in targets]

print("Final features used for multi-target model:")
print(final_features)
print("Feature count:", len(final_features))

def get_xy(data):
    X = data[final_features]
    y = data[targets]
    return X, y

X1, y1 = get_xy(iter1)
X2, y2 = get_xy(iter2)
X3, y3 = get_xy(iter3)
X_test, y_test = get_xy(test_df)

print(X1.shape, y1.shape)
print(X2.shape, y2.shape)
print(X3.shape, y3.shape)
print(X_test.shape, y_test.shape)

Final features used for multi-target model:
['visibility_km', 'last_updated_epoch', 'air_quality_Ozone', 'latitude', 'air_quality_PM10', 'condition_text', 'air_quality_Sulphur_dioxide', 'uv_index', 'air_quality_Nitrogen_dioxide', 'feels_like_celsius', 'precip_mm', 'cloud', 'pressure_mb', 'longitude', 'air_quality_Carbon_Monoxide', 'air_quality_PM2.5_lag1', 'air_quality_PM2.5_lag2', 'temperature_celsius_lag1', 'temperature_celsius_lag2', 'humidity_lag1', 'humidity_lag2']
Feature count: 21
(34320, 21) (34320, 3)
(34320, 21) (34320, 3)
(35360, 21) (35360, 3)
(26001, 21) (26001, 3)


In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor

In [21]:
def calculate_metrics_per_target(y_true, y_pred, target_names):
    rows = []

    for i, target in enumerate(target_names):
        mse = mean_squared_error(y_true.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true.iloc[:, i], y_pred[:, i])
        r2 = r2_score(y_true.iloc[:, i], y_pred[:, i])
        acc = r2 * 100

        rows.append([target, mse, rmse, mae, r2, acc])

    return pd.DataFrame(
        rows,
        columns=["Target", "MSE", "RMSE", "MAE", "R2", "Accuracy (%)"]
    ).round(3)

In [22]:
def get_xy(data):
    X = data[final_features]
    y = data[targets]
    return X, y

X1, y1 = get_xy(iter1)
X2, y2 = get_xy(iter2)
X3, y3 = get_xy(iter3)
X_test, y_test = get_xy(test_df)

In [24]:
from sklearn.linear_model import SGDRegressor

scaler = StandardScaler()

X1_s = scaler.fit_transform(X1)
X2_s = scaler.transform(X2)
X3_s = scaler.transform(X3)
X_test_s = scaler.transform(X_test)

model = MultiOutputRegressor(SGDRegressor(random_state=42))

# Iteration 1
model.partial_fit(X1_s, y1)
pred1 = model.predict(X1_s)

# Iteration 2
model.partial_fit(X2_s, y2)
pred2 = model.predict(X2_s)

# Iteration 3
model.partial_fit(X3_s, y3)
pred3 = model.predict(X3_s)

# Test
test_pred = model.predict(X_test_s)

sgd_iter = pd.concat([
    calculate_metrics_per_target(y1, pred1, targets).assign(Iteration="Iteration 1"),
    calculate_metrics_per_target(y2, pred2, targets).assign(Iteration="Iteration 2"),
    calculate_metrics_per_target(y3, pred3, targets).assign(Iteration="Iteration 3"),
])

sgd_iter["Model"] = "SGD"

sgd_test = calculate_metrics_per_target(y_test, test_pred, targets)
sgd_test["Model"] = "SGD"

In [25]:
from sklearn.ensemble import RandomForestRegressor

model = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42))

# Iteration 1
model.fit(X1, y1)
pred1 = model.predict(X1)

# Iteration 2
model.fit(pd.concat([X1, X2]), pd.concat([y1, y2]))
pred2 = model.predict(X2)

# Iteration 3
model.fit(pd.concat([X1, X2, X3]), pd.concat([y1, y2, y3]))
pred3 = model.predict(X3)

# Test
test_pred = model.predict(X_test)

rf_iter = pd.concat([
    calculate_metrics_per_target(y1, pred1, targets).assign(Iteration="Iteration 1"),
    calculate_metrics_per_target(y2, pred2, targets).assign(Iteration="Iteration 2"),
    calculate_metrics_per_target(y3, pred3, targets).assign(Iteration="Iteration 3"),
])

rf_iter["Model"] = "Random Forest"

rf_test = calculate_metrics_per_target(y_test, test_pred, targets)
rf_test["Model"] = "Random Forest"

In [26]:
import xgboost as xgb

pred1_list, pred2_list, pred3_list, test_list = [], [], [], []

for i in range(len(targets)):
    model = xgb.XGBRegressor(n_estimators=100, random_state=42)

    model.fit(X1, y1.iloc[:, i])
    p1 = model.predict(X1)

    model.fit(X2, y2.iloc[:, i], xgb_model=model.get_booster())
    p2 = model.predict(X2)

    model.fit(X3, y3.iloc[:, i], xgb_model=model.get_booster())
    p3 = model.predict(X3)

    pt = model.predict(X_test)

    pred1_list.append(p1)
    pred2_list.append(p2)
    pred3_list.append(p3)
    test_list.append(pt)

pred1 = np.column_stack(pred1_list)
pred2 = np.column_stack(pred2_list)
pred3 = np.column_stack(pred3_list)
test_pred = np.column_stack(test_list)

xgb_iter = pd.concat([
    calculate_metrics_per_target(y1, pred1, targets).assign(Iteration="Iteration 1"),
    calculate_metrics_per_target(y2, pred2, targets).assign(Iteration="Iteration 2"),
    calculate_metrics_per_target(y3, pred3, targets).assign(Iteration="Iteration 3"),
])

xgb_iter["Model"] = "XGBoost"

xgb_test = calculate_metrics_per_target(y_test, test_pred, targets)
xgb_test["Model"] = "XGBoost"

In [27]:
import lightgbm as lgb

pred1_list, pred2_list, pred3_list, test_list = [], [], [], []

for i in range(len(targets)):
    train1 = lgb.Dataset(X1, label=y1.iloc[:, i])
    train2 = lgb.Dataset(X2, label=y2.iloc[:, i])
    train3 = lgb.Dataset(X3, label=y3.iloc[:, i])

    model = lgb.train({}, train1, num_boost_round=100)
    p1 = model.predict(X1)

    model = lgb.train({}, train2, num_boost_round=100, init_model=model)
    p2 = model.predict(X2)

    model = lgb.train({}, train3, num_boost_round=100, init_model=model)
    p3 = model.predict(X3)

    pt = model.predict(X_test)

    pred1_list.append(p1)
    pred2_list.append(p2)
    pred3_list.append(p3)
    test_list.append(pt)

pred1 = np.column_stack(pred1_list)
pred2 = np.column_stack(pred2_list)
pred3 = np.column_stack(pred3_list)
test_pred = np.column_stack(test_list)

lgb_iter = pd.concat([
    calculate_metrics_per_target(y1, pred1, targets).assign(Iteration="Iteration 1"),
    calculate_metrics_per_target(y2, pred2, targets).assign(Iteration="Iteration 2"),
    calculate_metrics_per_target(y3, pred3, targets).assign(Iteration="Iteration 3"),
])

lgb_iter["Model"] = "LightGBM"

lgb_test = calculate_metrics_per_target(y_test, test_pred, targets)
lgb_test["Model"] = "LightGBM"

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008001 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4124
[LightGBM] [Info] Number of data points in the train set: 34320, number of used features: 21
[LightGBM] [Info] Start training from score 19.171962
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008217 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4025
[LightGBM] [Info] Number of data points in the train set: 34320, number of used features: 21
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008179 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3977
[LightGBM] [Info] Number of data points in the train set: 35360, number of used features: 21
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testi

In [28]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Input

scaler = StandardScaler()

X1_c = scaler.fit_transform(X1)
X2_c = scaler.transform(X2)
X3_c = scaler.transform(X3)
X_test_c = scaler.transform(X_test)

X1_c = X1_c.reshape((X1_c.shape[0], X1_c.shape[1], 1))
X2_c = X2_c.reshape((X2_c.shape[0], X2_c.shape[1], 1))
X3_c = X3_c.reshape((X3_c.shape[0], X3_c.shape[1], 1))
X_test_c = X_test_c.reshape((X_test_c.shape[0], X_test_c.shape[1], 1))

model = Sequential([
    Input(shape=(X1_c.shape[1], 1)),
    Conv1D(32, 2, activation='relu'),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(len(targets))
])

model.compile(optimizer='adam', loss='mse')

model.fit(X1_c, y1, epochs=10, verbose=0)
pred1 = model.predict(X1_c)

model.fit(X2_c, y2, epochs=10, verbose=0)
pred2 = model.predict(X2_c)

model.fit(X3_c, y3, epochs=10, verbose=0)
pred3 = model.predict(X3_c)

test_pred = model.predict(X_test_c)

cnn_iter = pd.concat([
    calculate_metrics_per_target(y1, pred1, targets).assign(Iteration="Iteration 1"),
    calculate_metrics_per_target(y2, pred2, targets).assign(Iteration="Iteration 2"),
    calculate_metrics_per_target(y3, pred3, targets).assign(Iteration="Iteration 3"),
])

cnn_iter["Model"] = "CNN"

cnn_test = calculate_metrics_per_target(y_test, test_pred, targets)
cnn_test["Model"] = "CNN"

1073/1073 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1073/1073 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
1105/1105 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
813/813 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [29]:
all_iterations = pd.concat([
    sgd_iter, rf_iter, xgb_iter, lgb_iter, cnn_iter
], ignore_index=True)

all_iterations

,Target,MSE,RMSE,MAE,R2,Accuracy (%),Iteration,Model
0,air_quality_PM2.5,1147.385,33.873,9.830,0.472,47.216,Iteration 1,SGD
1,temperature_celsius,2.533,1.592,1.183,0.954,95.398,Iteration 1,SGD
2,humidity,364.581,19.094,12.296,0.412,41.184,Iteration 1,SGD
3,air_quality_PM2.5,466.838,21.606,12.214,0.689,68.893,Iteration 2,SGD
4,temperature_celsius,2.407,1.551,1.033,0.977,97.717,Iteration 2,SGD
5,humidity,236.033,15.363,11.910,0.576,57.589,Iteration 2,SGD
6,air_quality_PM2.5,175.601,13.251,8.053,0.829,82.920,Iteration 3,SGD
7,temperature_celsius,2.013,1.419,1.013,0.963,96.297,Iteration 3,SGD
8,humidity,197.985,14.071,11.047,0.647,64.696,Iteration 3,SGD
9,air_quality_PM2.5,4.742,2.178,0.766,0.998,99.782,Iteration 1,Random Forest


In [31]:
all_test = pd.concat([
    sgd_test, rf_test, xgb_test, lgb_test, cnn_test
], ignore_index=True)

all_test

,Target,MSE,RMSE,MAE,R2,Accuracy (%),Model
0,air_quality_PM2.5,201.212,14.185,8.364,0.664,66.392,SGD
1,temperature_celsius,2.107,1.452,1.061,0.983,98.304,SGD
2,humidity,284.363,16.863,12.499,0.407,40.682,SGD
3,air_quality_PM2.5,47.711,6.907,2.977,0.920,92.031,Random Forest
4,temperature_celsius,0.859,0.927,0.581,0.993,99.308,Random Forest
5,humidity,173.713,13.180,9.313,0.638,63.763,Random Forest
6,air_quality_PM2.5,368.207,19.189,3.858,0.385,38.499,XGBoost
7,temperature_celsius,29.354,5.418,2.506,0.764,76.365,XGBoost
8,humidity,173.742,13.181,9.701,0.638,63.757,XGBoost
9,air_quality_PM2.5,75.664,8.699,2.886,0.874,87.362,LightGBM


In [32]:
rmse_table = all_test.pivot(index="Target", columns="Model", values="RMSE")
rmse_table

Model,CNN,LightGBM,Random Forest,SGD,XGBoost
Target,,,,,
air_quality_PM2.5,8.435,8.699,6.907,14.185,19.189
humidity,15.239,12.311,13.180,16.863,13.181
temperature_celsius,1.628,1.042,0.927,1.452,5.418


In [33]:
from google.colab import files

all_iterations.to_csv("training_iterations_results.csv", index=False)
all_test.to_csv("final_test_results.csv", index=False)

files.download("training_iterations_results.csv")
files.download("final_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>